In [ ]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import warnings
import torch
warnings.filterwarnings('ignore')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [ ]:
file_path = '../data/efsa_sentiment_classification.csv'
df = pd.read_csv(file_path)

df = df[:10]

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

INDUSTRY_LIST = df['Industry'].unique().tolist()

In [ ]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", trust_remote_code=True).half()
print("Model loaded successfully!")

Loading model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install accelerate`

In [ ]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
logger = logging.getLogger(__name__)


def query_model(model, tokenizer, prompt: str, max_new_tokens=512, temperature=0.7) -> str:
    """Generate model response for given prompt using Mistral Instruct format."""
    instruction = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
    try:
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                    temperature=temperature, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Remove prompt from response to get just the answer:
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""


def extract_company(text: str) -> str:
    """Extract company name from financial text."""
    prompt = (f"Financial text: {text}\n\n"
              "In the financial text above, which company is the described event associated with? "
              "Please only answer with the company name, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt)
        return response.strip() or "Unknown Company"
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return "Unknown Company"


def get_industry(text: str, company_name: str) -> str:
    """Get industry for a company using model prompt."""
    prompt = (f"Financial text: {text}\n"
              f"Company: {company_name}\n\n"
              f"What industry does {company_name} belong to? "
              f"Please provide the corresponding GICS industry sector from this list: {INDUSTRY_LIST}. "
              "Please only answer with the industry name, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt)
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"


def classify_coarse_grained_event(text: str, industry: str, company_name: str) -> Tuple[str, Any]:
    """Classify the coarse-grained event type."""
    coarse_events = list(FINANCIAL_EVENT_LIST.keys())
    prompt = (f"Financial text: {text}\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"What kind of events happened to {company_name}? "
              f"Please select the corresponding coarse-grained event from this list: {coarse_events}\n"
              "You must select exactly one event type from the given list. "
              "Please only answer with the coarse-grained event type, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt)
        return response.strip() or "Unknown Event", None
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event", None


def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str, history: Any) -> Tuple[str, Any]:
    """Classify the fine-grained event type."""
    if coarse_event in FINANCIAL_EVENT_LIST:
        fine_events = FINANCIAL_EVENT_LIST[coarse_event]
        prompt = (f"Financial text: {text}\n"
                  f"Company: {company_name}\n"
                  f"Industry: {industry}\n"
                  f"Coarse-grained event: {coarse_event}\n\n"
                  f"The coarse-grained event that occurred at {company_name} is {coarse_event}. "
                  f"Please select the corresponding fine-grained event from this list: {fine_events}\n"
                  "You must select exactly one event type from the given list. "
                  "Please only answer with the fine-grained event type, without additional text or punctuation."
                  )
    else:
        return "Unknown Fine Event", history

    try:
        response = query_model(model, tokenizer, prompt)
        return response.strip() or "Unknown Fine Event", history
    except Exception as e:
        return "Unknown Fine Event", history


def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str, history: Any) -> str:
    """Analyze sentiment polarity of the event."""
    prompt = (f"Financial text: {text}\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n"
              f"Coarse-grained event: {coarse_event}\n"
              f"Fine-grained event: {fine_event}\n\n"
              f"The coarse-grained event that occurred at {company_name} is {coarse_event}, "
              f"and the fine-grained event is {fine_event}. "
              f"Using the sentiment polarity options: {SENTIMENT_POLARITIES}, "
              "use your financial knowledge to identify the corresponding sentiment polarity label. "
              "Consider the impact on stock price and investor sentiment. "
              "Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt)
        return response.strip() or "NEU"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "NEU"

In [ ]:
def analyze_financial_text(text: str) -> Tuple[str, str, str, str, str]:
    """
    Analyze a single financial text and return the quintuple.

    Returns:
        Tuple[str, str, str, str, str]: (company_name, industry, coarse_event, fine_event, sentiment)
    """
    try:
        # Extract company name
        company_name = extract_company(text)

        # Get industry
        industry = get_industry(text, company_name)

        # Classify coarse-grained event
        coarse_event, history = classify_coarse_grained_event(
            text, industry, company_name)

        # Classify fine-grained event
        fine_event, history = classify_fine_grained_event(
            text, industry, company_name, coarse_event, history)

        # Analyze sentiment
        sentiment = analyze_sentiment(
            text, industry, company_name, coarse_event, fine_event, history)

        return company_name, industry, coarse_event, fine_event, sentiment

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return "Error", "Error", "Error", "Error", "Error"

In [ ]:
results = []
df = df[:100]
# Process each text
for idx, row in df.iterrows():
    text = row['Sentence']

    if pd.isna(text) or text.strip() == '':
        logger.warning(f"Empty text at index {idx}")
        results.append(("", "", "", "", ""))
        continue

    print(f"Processing {idx+1}/{len(df)}: {text[:50]}...")

    # Analyze the text
    company_name, industry, coarse_event, fine_event, sentiment = analyze_financial_text(
        text)

    # Store results
    results.append(
        (company_name, industry, coarse_event, fine_event, sentiment))

    # Print progress every 10 items
    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1}/{len(df)} texts")

In [ ]:
results_df = pd.DataFrame(results, columns=[
                          'Company Name', 'Industry', 'Coarse Event', 'Fine Event', 'Sentiment'])
results_df.head(10)

2025-07-15 21:13:45,283 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,288 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,289 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,289 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,290 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,290 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,290 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,291 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,291 - ERROR - Error extracting company name:

Processing 1/10: Royal Mail chairman Donald Brydon set to step down...
Processing 2/10: Slump in Weir leads FTSE down from record high...
Processing 3/10: AstraZeneca wins FDA approval for key new lung can...
Processing 4/10: UPDATE 1-Lloyds to cut 945 jobs as part of 3-year ...
Processing 5/10: Standard Chartered Shifts Emerging-Markets Strateg...
Processing 6/10: AstraZeneca's Lung Cancer Drug Tagrisso Gets FDA A...
Processing 7/10: Severn Trent share price rises as first half profi...
Processing 8/10: Glencore sees Tripoli-based NOC as sole legal sell...
Processing 9/10: Lloyds to cut 945 jobs as part of three-year restr...
Processing 10/10: AstraZeneca to Buy ZS Pharma for $2.7 Billion...
Completed 10/10 texts


In [ ]:
df

In [ ]:
def calculate_accuracy(predicted_df, ground_truth_df, predicted_label_column, ground_truth_label_column):
    """Calculates the accuracy for a specific label column."""
    predicted_values = predicted_df[predicted_label_column].reset_index(
        drop=True)
    ground_truth_values = ground_truth_df[ground_truth_label_column].reset_index(
        drop=True)

    correct_predictions = (predicted_values == ground_truth_values).sum()
    total_predictions = len(predicted_df)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    return accuracy


# Calculate accuracy for each label
labels = ['Company Name', 'Industry',
          'Coarse Event', 'Fine Event', 'Sentiment']
accuracy_results = {}

for label in labels:
    # Handle different column names for ground truth
    if label == 'Company Name':
        ground_truth_column = 'Company'
    elif label == 'Coarse Event':
        ground_truth_column = 'Coarse-Grained Event'
    elif label == 'Fine Event':
        ground_truth_column = 'Fine-Grained Event'
    elif label == 'Industry':
        ground_truth_column = 'Industry'
    elif label == 'Sentiment':
        ground_truth_column = 'Sentiment'

    if ground_truth_column in df.columns:
        accuracy = calculate_accuracy(
            results_df, df, predicted_label_column=label, ground_truth_label_column=ground_truth_column)
        accuracy_results[label] = accuracy

print("Accuracy for each label:")
for label, accuracy in accuracy_results.items():
    print(f"{label}: {accuracy:.2f}" if isinstance(
        accuracy, float) else f"{label}: {accuracy}")